In [ ]:
import pandas as pd


In [ ]:
df=pd.read_csv("emails.csv")

In [ ]:
df.sample(10)

In [ ]:
df['Prediction'].value_counts()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
df.dtypes.value_counts()

In [ ]:
neg_count = (df.drop(columns=['Email No.', 'Prediction']) < 0).sum().sum()
print("Negative values found:", neg_count)

In [ ]:
X = df.drop(columns=['Email No.', 'Prediction'])
y = df['Prediction']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [ ]:
from sklearn.preprocessing import Normalizer, StandardScaler

knn_scaler = Normalizer()
X_train_knn = knn_scaler.fit_transform(X_train)
X_test_knn = knn_scaler.transform(X_test)

svm_scaler = StandardScaler()
X_train_svm = svm_scaler.fit_transform(X_train)
X_test_svm = svm_scaler.transform(X_test)


In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=300, random_state=42)

X_train_knn = svd.fit_transform(X_train_knn)
X_test_knn = svd.transform(X_test_knn)

X_train_svm = svd.fit_transform(X_train_svm)
X_test_svm = svd.transform(X_test_svm)


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')

knn.fit(X_train_knn, y_train)

y_pred_knn = knn.predict(X_test_knn)

print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn))


In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='linear', class_weight='balanced', random_state=42)

svm.fit(X_train_svm, y_train)

y_pred_svm = svm.predict(X_test_svm)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))


In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

svm_probs = svm.decision_function(X_test_svm)
knn_probs = knn.predict_proba(X_test_knn)[:, 1]

fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_probs)
fpr_knn, tpr_knn, _ = roc_curve(y_test, knn_probs)

plt.plot(fpr_svm, tpr_svm, label='SVM (AUC = {:.2f})'.format(auc(fpr_svm, tpr_svm)))
plt.plot(fpr_knn, tpr_knn, label='KNN (AUC = {:.2f})'.format(auc(fpr_knn, tpr_knn)))
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

knn_params = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'metric': ['euclidean', 'manhattan', 'cosine'],
    'weights': ['uniform', 'distance']
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid=knn_params,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1
)

grid_knn.fit(X_train_knn, y_train)

print("Best KNN Params:", grid_knn.best_params_)
print("Best KNN Score:", grid_knn.best_score_)


In [ ]:
from sklearn.svm import SVC

svm_params = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

grid_svm = GridSearchCV(
    SVC(class_weight='balanced'),
    param_grid=svm_params,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1
)

grid_svm.fit(X_train_svm, y_train)

print("Best SVM Params:", grid_svm.best_params_)
print("Best SVM Score:", grid_svm.best_score_)


In [ ]:
best_knn = grid_knn.best_estimator_

from sklearn.metrics import classification_report, accuracy_score

y_pred_knn = best_knn.predict(X_test_knn)

print("KNN Test Accuracy:", accuracy_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn))


In [ ]:
best_svm = grid_svm.best_estimator_

from sklearn.metrics import classification_report, accuracy_score

y_pred_svm = best_svm.predict(X_test_svm)

print("SVM Test Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))
